In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

True

In [3]:
access_key = os.getenv("Access_key_ID")
secret_key = os.getenv("Secret_access_key")
bucket = os.getenv("BUCKET_NAME")
region = os.getenv("REGION_NAME")

In [4]:
spark = (
    SparkSession.builder.appName("S3DataTransformation")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.1")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.access.key", access_key)
    .config("spark.hadoop.fs.s3a.secret.key", secret_key)
    .config("spark.hadoop.fs.s3a.endpoint", "s3.amazonaws.com")
    .config("spark.hadoop.fs.s3a.region", region)
    .getOrCreate()
)

25/04/17 09:19:28 WARN Utils: Your hostname, brempong-HP-EliteBook-840-G7-Notebook-PC resolves to a loopback address: 127.0.1.1; using 192.168.36.43 instead (on interface wlp0s20f3)
25/04/17 09:19:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/brempong/Ecommerce-DataLakehouse/venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/brempong/.ivy2/cache
The jars for the packages stored in: /home/brempong/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e6e0200b-f389-4d12-8c11-6fe1f3908d99;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.1 in central
	found com.amazonaws#aws-java-sdk-bundle;1.11.901 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 239ms :: artifacts dl 11ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.11.901 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.1 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	----------------------------

In [5]:
spark

In [6]:
folder_path = f"s3a://{bucket}/raw-data/orders_apr_2025/"
df = spark.read.csv(folder_path, header=True, inferSchema=True)
df.show()

25/04/17 09:19:33 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+---------+--------+-------+-------------------+------------+----------+
|order_num|order_id|user_id|    order_timestamp|total_amount|      date|
+---------+--------+-------+-------------------+------------+----------+
|       11|   13000|   6260|2025-04-07 10:44:00|      444.48|2025-04-07|
|       40|   13001|   4436|2025-04-07 16:59:00|      231.16|2025-04-07|
|       44|   13002|   1271|2025-04-07 21:29:00|      245.62|2025-04-07|
|       43|   13003|   9974|2025-04-07 03:01:00|      147.55|2025-04-07|
|       54|   13004|   7561|2025-04-07 11:09:00|      137.04|2025-04-07|
|       35|   13005|   8784|2025-04-07 01:02:00|      150.17|2025-04-07|
|       80|   13006|   8694|2025-04-07 05:20:00|      162.56|2025-04-07|
|       95|   13007|   6024|2025-04-07 01:36:00|      237.47|2025-04-07|
|       85|   13008|   5508|2025-04-07 19:36:00|      110.47|2025-04-07|
|       71|   13009|   9916|2025-04-07 16:55:00|      146.99|2025-04-07|
|       31|   13010|   6641|2025-04-07 09:39:00|   

In [7]:
df.select("date").distinct().show()

+----------+
|      date|
+----------+
|2025-04-07|
|2025-04-03|
|2025-04-12|
|2025-04-14|
|2025-04-08|
|2025-04-01|
|2025-04-10|
|2025-04-06|
|2025-04-02|
|2025-04-09|
|2025-04-13|
|2025-04-15|
|2025-04-04|
|2025-04-11|
|2025-04-05|
+----------+



In [10]:
# check for missing values
df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+---------+--------+-------+---------------+------------+----+
|order_num|order_id|user_id|order_timestamp|total_amount|date|
+---------+--------+-------+---------------+------------+----+
|        0|       0|      0|              0|           0|   0|
+---------+--------+-------+---------------+------------+----+



In [11]:
df = df.withColumn("order_time", date_format(col("order_timestamp"), "HH:mm:ss"))
df.show()

+---------+--------+-------+-------------------+------------+----------+----------+
|order_num|order_id|user_id|    order_timestamp|total_amount|      date|order_time|
+---------+--------+-------+-------------------+------------+----------+----------+
|       11|   13000|   6260|2025-04-07 10:44:00|      444.48|2025-04-07|  10:44:00|
|       40|   13001|   4436|2025-04-07 16:59:00|      231.16|2025-04-07|  16:59:00|
|       44|   13002|   1271|2025-04-07 21:29:00|      245.62|2025-04-07|  21:29:00|
|       43|   13003|   9974|2025-04-07 03:01:00|      147.55|2025-04-07|  03:01:00|
|       54|   13004|   7561|2025-04-07 11:09:00|      137.04|2025-04-07|  11:09:00|
|       35|   13005|   8784|2025-04-07 01:02:00|      150.17|2025-04-07|  01:02:00|
|       80|   13006|   8694|2025-04-07 05:20:00|      162.56|2025-04-07|  05:20:00|
|       95|   13007|   6024|2025-04-07 01:36:00|      237.47|2025-04-07|  01:36:00|
|       85|   13008|   5508|2025-04-07 19:36:00|      110.47|2025-04-07|  19

In [12]:
partitioned_df = df.repartition(15, col("date")).sortWithinPartitions(col("order_time"))
partitioned_df.show()

+---------+--------+-------+-------------------+------------+----------+----------+
|order_num|order_id|user_id|    order_timestamp|total_amount|      date|order_time|
+---------+--------+-------+-------------------+------------+----------+----------+
|       53|   13022|   3939|2025-04-07 00:12:00|       35.57|2025-04-07|  00:12:00|
|       27|   13073|   2234|2025-04-07 00:14:00|      332.58|2025-04-07|  00:14:00|
|       67|   13404|   5533|2025-04-07 00:18:00|      398.57|2025-04-07|  00:18:00|
|       60|   13047|   6775|2025-04-07 00:24:00|       37.55|2025-04-07|  00:24:00|
|       64|   13212|   4065|2025-04-07 00:32:00|      432.23|2025-04-07|  00:32:00|
|       95|   13017|   4043|2025-04-07 00:35:00|       493.5|2025-04-07|  00:35:00|
|       57|   13230|   7890|2025-04-07 00:38:00|      197.07|2025-04-07|  00:38:00|
|       47|   13025|   5455|2025-04-07 00:40:00|      166.87|2025-04-07|  00:40:00|
|       79|   13232|   9547|2025-04-07 00:42:00|      112.31|2025-04-07|  00

In [15]:
df.select("order_time").distinct().show()

+----------+
|order_time|
+----------+
|  00:41:00|
|  02:06:00|
|  13:29:00|
|  14:48:00|
|  04:43:00|
|  08:10:00|
|  17:42:00|
|  15:37:00|
|  10:44:00|
|  01:41:00|
|  15:23:00|
|  16:58:00|
|  01:47:00|
|  20:53:00|
|  19:03:00|
|  14:53:00|
|  13:42:00|
|  14:18:00|
|  14:36:00|
|  22:56:00|
+----------+
only showing top 20 rows

